In [1]:
%run_nb spark-start --data-format delta

Args: Namespace(data_format='delta', port_offset=2) - unknown_args: []
Spark version: 4.1.2, Driver memory: 16g, Executor memory: 8g, Service: jupyter-spark-4.1, Data format: delta
Spark packages: io.delta:delta-spark_4.1_2.13:4.1.0
Spark extensions: io.delta.sql.DeltaSparkSessionExtension
Spark catalog configs: {'spark.sql.catalog.spark_catalog': 'org.apache.spark.sql.delta.catalog.DeltaCatalog'}


,catalog
0,spark_catalog


Version,4.1.2
Master,local[2]
AppName,main


               total        used        free      shared  buff/cache   available
Mem:            62Gi        18Gi       6.1Gi       658Mi        38Gi        43Gi
Swap:          8.0Gi          0B       8.0Gi


In [6]:
spark

Version,4.1.2
Master,local[2]
AppName,main


In [7]:
import great_expectations as gx

df = spark.createDataFrame(
    [
        (1, "Hello, World!"),
        (2, "Hello, World!"),
    ],
    ["id", "message"],
)

df.show()

+---+-------------+
| id|      message|
+---+-------------+
|  1|Hello, World!|
|  2|Hello, World!|
+---+-------------+



In [8]:
# ---------------------------------------------------------
# 2. Create the Great Expectations objects
# ---------------------------------------------------------

context = gx.get_context()

data_source = context.data_sources.add_spark(
    name="hello_world_source"
)

data_asset = data_source.add_dataframe_asset(
    name="hello_world_asset"
)

batch_definition = data_asset.add_batch_definition_whole_dataframe(
    "hello_world_batch"
)

# Pass the in-memory Spark DataFrame to GX
batch = batch_definition.get_batch(
    batch_parameters={"dataframe": df}
)

# ---------------------------------------------------------
# 3. Validate the DataFrame
# ---------------------------------------------------------

expectation = gx.expectations.ExpectColumnValuesToBeInSet(
    column="message",
    value_set=["Hello, World!"],
)

result = batch.validate(expectation)

print("Validation successful:", result.success)

assert result.success, "Data quality validation failed"

spark.stop()

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpf80z5cq7' for ephemeral docs site


Calculating Metrics:   0%|          | 0/13 [00:00<?, ?it/s]

Validation successful: True
